# Open Notebook in Colab


[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/childmindresearch/llm_tracker/blob/main/tutorials/anonymization_tutorial.ipynb)

# Anonymizing PII Before LLM Coding

`llm_tracker` sends your documents to an LLM API, so **anything identifiable in them
leaves your machine**. The privacy note at the top of [`tutorial.ipynb`](../tutorial.ipynb)
recommends removing PII locally *before* any analysis. This notebook is the companion
walkthrough for doing exactly that with
[`anonymize-pii`](https://github.com/childmindresearch/anonymize-pii), a local pipeline
built on [Microsoft Presidio](https://github.com/microsoft/presidio).

Everything in this notebook runs locally. No text is sent anywhere until the last
section, where the anonymized output is handed back to `llm_tracker`.

**What you will do:**

1. Install `anonymize-pii` and its models.
2. Convert your documents into the `{id: text}` JSON format it expects.
3. Run the anonymizer.
4. **Audit the results** — inspect what was flagged, what was replaced, and what slipped
   through. This step is not optional.
5. Tune precision with skiplists so clinical vocabulary is not redacted.
6. Feed the anonymized text into `llm_tracker`.

**Prerequisites:**

- Python **3.12 or newer** (`anonymize-pii` sets `requires-python = ">=3.12"`) and `git`.
- Several GB of free disk: the pipeline pulls PyTorch plus three NER models.
- Optional: a GPU. The pipeline uses CUDA when `torch.cuda.is_available()`, otherwise CPU.
  On CPU expect roughly seconds-to-minutes per document depending on length.

> ⚠️ **No automated tool removes 100% of PII.** Treat the output as a first pass that
> still needs human review, and keep the un-anonymized source out of any directory you
> sync, share, or commit.

## How the pipeline works

Understanding the shape of the pipeline makes the outputs (and its failure modes) much
easier to read.

**1. Three detectors scan every document independently.** `configs` in
`src/anonymize_pii/config.py` defines them:

| engine | model | notes |
| --- | --- | --- |
| `spacy` | `en_core_web_lg` | fast statistical NER |
| `stanza` | `en` | separate architecture, so it catches different spans |
| `GLiNER` | `nvidia/gliner-pii` | prompt-based NER, run at `threshold=0.5` on 512-character chunks |

The `spacy` and `stanza` engines also load Presidio's predefined rule-based recognizers
(emails, phone numbers, credit cards, US identifiers), so the run mixes model-based and
regex-based detection. The entity types being tracked are the `Entities` list in
`config.py`.

**2. Findings are merged, highest confidence wins.** `process_full_document()` in
`anonymizers.py` pools the hits from all three engines; when several engines flag the
same string, the entity type from the highest-scoring detection is kept. Detection is
therefore a **union** — a span only needs one engine to catch it.

**3. A filter drops known-safe terms.** `PIIFilter` (`helpers.py`) discards any flagged
span that is in the skiplist, contains a time word (`day`, `morning`, `age`…), or
contains a general term (`DSM-5`, `zoom`…). This is where you stop clinical vocabulary
from being redacted — see *Tuning precision* below.

**4. Replacement is deny-list based.** `AnonymizeText()` builds a Presidio
`PatternRecognizer` from the surviving strings and re-scans the document, so **every**
occurrence of a flagged string is replaced, not just the one that was detected. The
important consequence: anything the detectors missed remains in the text **verbatim**.
That is exactly what the audit step looks for.

## 1. Install

Clone the repository next to this notebook. `anonymize-pii` is run as a script from a
checkout, not installed as a library.

In [ ]:
!git clone https://github.com/childmindresearch/anonymize-pii.git

### Option A — `uv` (recommended)

The repo ships a `uv.lock` and declares the spaCy model wheels plus `headhunter` under
`[tool.uv.sources]`, so `uv sync` reproduces the environment exactly, models included.

In [ ]:
# Install uv if you don't have it (skip otherwise):
!curl -LsSf https://astral.sh/uv/install.sh | sh

!cd anonymize-pii && uv sync

### Option B — `pip`

`requirements.txt` pins the same versions and includes the spaCy model wheels, but it
**omits `headhunter`**, which is needed only for the optional `--parse` step (last
section). Install it separately if you want that step.

In [ ]:
!pip install -r anonymize-pii/requirements.txt

# Only needed for the optional --parse step:
!pip install "git+https://github.com/childmindresearch/headhunter"

Two more model downloads happen automatically on the **first run**, then stay cached:

- Stanza's English models → `~/stanza_resources`
- the GLiNER weights `nvidia/gliner-pii` → your Hugging Face cache

Now set the paths used throughout the notebook.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

ANON_DIR = Path("anonymize-pii").resolve()      # the checkout you just cloned
SRC_DIR = ANON_DIR / "src" / "anonymize_pii"    # main.py has to run from here (see below)
RAW_DIR = ANON_DIR / "data" / "raw"             # inputs
EXPORT_DIR = ANON_DIR / "data" / "exports"      # outputs

# Interpreter that has the dependencies. With `uv sync` (Option A) it is the project venv;
# with pip (Option B) it is whatever runs this notebook.
PYTHON = str(ANON_DIR / ".venv" / "bin" / "python")   # Windows: .venv/Scripts/python.exe
# PYTHON = sys.executable                             # uncomment for Option B / Colab

print(ANON_DIR)

## 2. Prepare your input

The anonymizer reads a single JSON file, `data/raw/Reports.json`, shaped as
`{document_id: "full text of the document"}`:

```json
{
  "patient1": "Clinical Observation Report ...",
  "patient2": "Intake Summary ..."
}
```

Those keys become the document IDs in every output file, so use the same IDs you intend
to use in `llm_tracker` — that is what lets you line codings up with the original records
later.

If your data is a CSV (the usual `llm_tracker` starting point), convert it:

In [ ]:
df = pd.read_csv("my_documents.csv")   # one document per row

reports = dict(
    zip(df["doc_id"].astype(str), df["text"].astype(str))
)

RAW_DIR.mkdir(parents=True, exist_ok=True)
with open(RAW_DIR / "Reports.json", "w", encoding="utf-8") as fh:
    json.dump(reports, fh, ensure_ascii=False, indent=2)

print(f"{len(reports)} reports written to {RAW_DIR / 'Reports.json'}")

**Only put free text through the anonymizer.** Structured columns with controlled
vocabularies (sex, diagnosis codes, city names, response scales) should be handled with
your own de-identification rules instead. Running NER over them corrupts the values
without buying any privacy — see the language section below.

**Just want to try it?** The repo ships a synthetic report set (fake names, fake
addresses, fake phone numbers) you can use instead:

In [ ]:
import shutil

RAW_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy(ANON_DIR / "tests" / "Reports.json", RAW_DIR / "Reports.json")

sample = json.loads((RAW_DIR / "Reports.json").read_text(encoding="utf-8"))
print(f"{len(sample)} sample reports: {list(sample)}")
print(sample[list(sample)[0]][:400])

## 3. Run the anonymizer

⚠️ **`main.py` must be run from `src/anonymize_pii/`.** `config.py` resolves every data
path from the current working directory (`root_dir = Path.cwd().parents[1]`) and the
modules import each other by bare name (`from config import ...`). Launched from anywhere
else it fails to import or writes to the wrong place.

From a terminal that is:

```bash
cd anonymize-pii/src/anonymize_pii
uv run python main.py          # or: python main.py
```

From this notebook, pass `cwd=` explicitly:

In [ ]:
result = subprocess.run(
    [PYTHON, "main.py", "--mask", "entity", "--output", "merged"],
    cwd=SRC_DIR,            # required — config.py derives all paths from the cwd
    capture_output=True,
    text=True,
)

print(result.stdout[-3000:])
if result.returncode != 0:
    print("--- stderr ---")
    print(result.stderr[-3000:])

### Options

| flag | default | effect |
| --- | --- | --- |
| `--mask entity` | ✅ | replace each span with its entity type: `<PERSON>`, `<LOCATION>`, `<PHONE_NUMBER>`… |
| `--mask redact` | | replace every detected span with a single `<REDACTED>` label |
| `--output merged` | ✅ | three JSON files in `data/exports`, covering all documents |
| `--output single` | | one subdirectory per document ID under `data/exports`, with `Anonymized_Report.json` (singular) plus a copy of the source as `Original_Report.json` |
| `--parse` | | pre-process the input with `headhunter` first (see the last section) |

Keep `--mask entity` while you are still reviewing: knowing *what kind* of entity was
removed makes the output far easier to audit, and downstream LLM coding reads
`<PERSON> reported ...` more naturally than `<REDACTED> reported ...`. Switch to
`--mask redact` if the entity types are themselves too revealing.

## 4. Read and audit the output

`--output merged` writes three files to `data/exports`:

| file | contents |
| --- | --- |
| `Anonymized_Reports.json` | `{id: anonymized_text}` — same shape as the input |
| `Iterator.json` | every entity each engine flagged, with type and confidence, plus the merged `Deny` map |
| `PII_Log.json` | one record per replacement: entity type, `start`/`end` offsets, score, and which recognizer fired |

In [ ]:
with open(EXPORT_DIR / "Anonymized_Reports.json", encoding="utf-8") as fh:
    anonymized = json.load(fh)
with open(EXPORT_DIR / "Iterator.json", encoding="utf-8") as fh:
    iterator = json.load(fh)
with open(EXPORT_DIR / "PII_Log.json", encoding="utf-8") as fh:
    pii_log = json.load(fh)

first_id = next(iter(anonymized))
print(f"=== {first_id} ===")
print(anonymized[first_id][:1500])

### What was flagged, and how confidently

Flatten `Iterator.json` and look at the **lowest** scores first — those are the likeliest
false positives, i.e. real content about to be destroyed.

In [ ]:
ENGINES = ["spacy", "stanza", "GLiNER"]

rows = [
    {
        "doc_id": doc_id,
        "engine": engine,
        "text": text,
        "entity_type": entity_type,
        "score": score,
    }
    for doc_id, per_engine in iterator.items()
    for engine in ENGINES
    for text, (entity_type, score) in per_engine.get(engine, {}).items()
]

flagged = pd.DataFrame(rows).sort_values("score")
print(f"{len(flagged)} detections, {flagged['text'].nunique()} distinct strings")
flagged.head(30)

### Which detections only one engine agreed on

Because detection is a union of three engines, a span found by only one of them is where
disagreement lives. Scan these for both kinds of error: false positives to add to your
skiplist, and near-misses that hint at PII the others also missed.

In [ ]:
agreement = (
    flagged.groupby(["doc_id", "text"])
    .agg(n_engines=("engine", "nunique"), max_score=("score", "max"),
         types=("entity_type", lambda s: sorted(set(s))))
    .reset_index()
    .sort_values(["n_engines", "max_score"])
)

agreement.head(30)

### Confirm the strings you know about are gone

The single most valuable check: take identifiers you *know* are in the source and confirm
they no longer appear anywhere in the output. Replacement is deny-list based, so a missed
detection means the string survives untouched.

In [ ]:
# Replace with identifiers you know appear in your own source documents.
must_not_appear = ["Aisha Patel", "323-555-0891", "281 Pleasant Boulevard"]

for needle in must_not_appear:
    hits = [doc_id for doc_id, text in anonymized.items() if needle in text]
    print(f"{needle!r}: {'STILL PRESENT in ' + str(hits) if hits else 'not found ✓'}")

### Per-replacement log

`PII_Log.json` records the character offsets of each replacement against the source
document, which is what you need to diff the original against the anonymized version.

In [ ]:
log_rows = [
    {"doc_id": doc_id, **entry}
    for doc_id, entries in pii_log.items()
    for entry in entries
]

pd.DataFrame(log_rows).head(20)

## 5. Tune precision

Three knobs, in the order you will reach for them.

**Skiplist (no code changes).** `load_skiplist_from_directory()` merges **every `.txt`
file** in `data/external` — currently `skiplist.txt` and `ICD11.txt` — one term per line.
Dropping in your own file is enough; you never have to edit theirs. Matching is
case-insensitive against the **whole** flagged span (`PIIFilter.check_skiplist`), so list
terms exactly as the detector reports them in `Iterator.json` — a partial phrase will not
match.

In [ ]:
# Terms your run flagged that are clinical vocabulary, not PII.
extra_terms = ["Bipolar", "Ritalin", "Hollywood Hills Elementary"]

(ANON_DIR / "data" / "external" / "project_skiplist.txt").write_text(
    "\n".join(extra_terms) + "\n", encoding="utf-8"
)

**Filter word lists (`config.py`).** `timewords` and `generalwords` drop any span
*containing* one of those words, which is a blunter instrument than the skiplist — useful
for recurring patterns like date fragments or product names.

**Entity types (`config.py`).** The `Entities` list controls what is looked for at all.
Trimming it (e.g. removing `DATE_TIME` when dates are analytically necessary, or the
`US_*` recognizers when they only produce noise on non-US data) raises precision;
extending it only helps for entity types the models actually support.

After changing any of these, re-run step 3 and re-audit. Iterating two or three times is
normal.

## 6. Optional: parse documents first (`--parse`)

With `--parse`, the input is normalized by
[`headhunter`](https://github.com/childmindresearch/headhunter) before anonymization,
which is what you want when you need to **anonymize only certain sections** of long,
inconsistently formatted reports, or when your input is tabular. Configure it via
`headhunter_config` in `config.py`; the parsed intermediate is written to
`data/parsed/Parsed_Reports.json`.

The mode is inferred from the input:

| `input_path` | `content_columns` | mode |
| --- | --- | --- |
| `.json` (`{id: text}`) | ignored | JSON |
| `.csv` / `.parquet` | exactly one | single-column |
| `.csv` / `.parquet` | several | multi-column |

```python
headhunter_config = {
    'input_path': str(report_in / 'my_reports.csv'),
    'content_columns': ['report'],
    'id_column': 'report_id',
    'parser_config': {'heading_max_words': 10},
    'expected_headings': None,
    'match_threshold': 80,
    'headings_to_anonymize': ['clinical summary', 'treatment plan'],
    'separate_headings_into_reports': False,
}
```

Notes worth knowing before you spend time on it:

- An empty or missing `headings_to_anonymize` means the whole document is anonymized.
- A non-empty one keeps only the matching heading subtrees (hierarchy preserved), so
  **content under other headings is dropped from the output** — it is a filter, not a
  mask.
- `separate_headings_into_reports=True` emits one entry per matched section, keyed
  `{id}/{heading}`, instead of merging them into one document.
- `parser_config`, `expected_headings` and `match_threshold` apply to JSON and
  single-column mode only; they are ignored in multi-column mode.

## 7. Working in a language other than English

The defaults are English-only. Adapting the pipeline to another language is a handful of
edits, but recall behaves differently enough that it is worth knowing what to expect. The
notes below come from running the pipeline over Spanish-language clinical records.

**Install the target-language spaCy model** (the pinned wheels are English):

```bash
uv run python -m spacy download es_core_news_lg
uv run python -m spacy download es_core_news_sm     # used by the GLiNER config
```

**Point the three engines at that language** in `config.py`:

```python
spacy = {'name': 'spacy', 'config': {
    "nlp_engine_name": "spacy",
    "models": [{"lang_code": "es", "model_name": "es_core_news_lg"}]}}

stanza = {'name': 'stanza', 'config': {
    "nlp_engine_name": "stanza",
    "models": [{"lang_code": "es", "model_name": "es"}]}}

GLiNER = {'name': 'GLiNER',
    'config': {"nlp_engine_name": "spacy",
               "models": [{"lang_code": "es", "model_name": "es_core_news_sm"}]},
    'external_model': "nvidia/gliner-pii"}
```

**Register recognizers for that language** in `get_warm_engines()`, and declare it on the
GLiNER recognizer. Presidio matches recognizers against the requested language, so
without these two changes the run either errors out or finds nothing:

```python
registry.load_predefined_recognizers(languages=["es"])          # instead of ()

gliner_rec = GlinerRecognizer(
    model_name=config.get('external_model'),
    labels=Entities,
    device=device,
    supported_language="es",                                     # added
)
```

**Set the scan language** in `EntityScanner.scan()` (`anonymizers.py`):

```python
results = self.analyzer.analyze(text=chunk, language="es", entities=self.entities)
```

**Leave the `language="en"` inside `AnonymizeText()` as it is.** That second pass is a
deny-list lookup with a `PatternRecognizer`, not NER, and Presidio registers
`PatternRecognizer` as English by default — changing it there breaks the replacement step
without improving detection.

### What to expect

- **Presidio's built-in identifier recognizers are US-specific** (`US_SSN`,
  `US_DRIVER_LICENSE`, `US_PASSPORT`). National ID numbers elsewhere have no built-in
  recognizer, so add a `PatternRecognizer` or a deterministic regex post-processing pass
  for them and for local phone-number formats. Do not assume the NER covers them.
- **ALL-CAPS text breaks name detection**, and administrative records are full of it.
  Section 8 measures this on public data and builds a rule-based pass, keyed on titles
  (`Dra.`, `Lic.`) and field labels (`Paciente:`), that recovers what the models lose.
- **Do not title-case the text to boost recall.** It does help the models find more
  names, but it produces far more false positives than it fixes (also measured in
  section 8).
- **Only run it over free-text fields.** On controlled-vocabulary columns the models
  produce confident nonsense — a sex value flagged as `<PERSON>`, a diagnosis label as
  `<ORGANIZATION>` — destroying the column with no privacy gain. De-identify structured
  fields with explicit rules instead.

## 8. Recovering names the models miss

Person names are the hardest entity to get right, and the failure is asymmetric: a missed
name stays in the text verbatim, while a false positive only costs you a word of content.
This section builds a small rule-based pass that runs *after* the models and closes the
most common gaps.

Everything below runs on the public Reddit sample bundled with this repo — no sensitive
data required — plus a short synthetic record with invented names to exercise the
record-style patterns.

**First, the honest version of the problem.** It is often claimed that NER truncates long
names, e.g. that it catches only the first of two given names. On well-formed mixed-case
text that is mostly **not** what happens — `es_core_news_lg` returns the whole span:

```
"Se comunicó con María José García Pérez, madre del paciente."
    → PER: ['María José García Pérez']              ✓ all four tokens
"Refiere que Ana Paula Quispe Mamani la acompañó a la guardia."
    → PER: ['Ana Paula Quispe Mamani']              ✓
```

The real failures are these three:

**1. ALL-CAPS text inverts the labels.** Administrative records are full of upper-case
fields, and the models fall apart on them:

```
"SE COMUNICO CON MARIA JOSE GARCIA PEREZ, MADRE DEL PACIENTE."
    → LOC: ['SE COMUNICO', 'CON MARIA JOSE GARCIA PEREZ']   ✗ the name is a LOCATION
    → PER: ['MADRE DEL PACIENTE']                           ✗ and a non-name is the PERSON
```

The name is still redacted here — `anonymize-pii` replaces every entity type it tracks, so
a name mislabelled `LOCATION` is masked anyway. What you lose is the ability to trust the
entity types, and any span the models drop entirely.

**2. Field layouts make spans bleed across line breaks.** In `Paciente: <name>\nAcompañante:
<name>`, the span comes back as `María José García Pérez\nAcompañante` — the label of the
*next* field is now part of the "name".

**3. Titles get absorbed.** `La Lic. Ana Lucía Ramírez ...` returns
`Lic. Ana Lucía Ramírez`, title included.

### 8.1 Measure it on public data

Load the Reddit sample that ships with `llm_tracker` (60 posts from r/anxiety,
r/depression and r/autism).

In [ ]:
reddit = pd.read_csv("../sample_data/reddit_autism_anxiety_depression.csv")

# Fallback if you are running outside a repo checkout (e.g. a bare Colab runtime):
# reddit = pd.read_csv(
#     "https://mair.sites.fas.harvard.edu/datasets/rmhd_27subreddits_1300posts_train.csv",
#     index_col=0,
# )

posts = reddit["post"].astype(str).tolist()
print(f"{len(posts)} posts, {sum(len(p) for p in posts):,} characters")

Run spaCy over them directly. (The demo cells in this section need spaCy in *this* kernel:
`pip install spacy` and `python -m spacy download en_core_web_lg`. In a real run you would
instead read the `PERSON` entries out of `Iterator.json` — the pipeline already did this
work; see 8.4.)

In [ ]:
import spacy

nlp = spacy.load("en_core_web_lg")
docs = list(nlp.pipe(posts))

person_spans = sorted({e.text for d in docs for e in d.ents if e.label_ == "PERSON"})
print(f"{len(person_spans)} distinct PERSON spans:")
print(person_spans)

On this sample that returns nine distinct spans:

```
['Asperger', 'Brandon Sanderson', 'Nicholas Hoult', 'Sanderson', 'Sid',
 'Skins', 'Yada Yada', 'aspergers', 'dyspraxia']
```

Four of the nine — `Asperger`, `aspergers`, `dyspraxia`, `Skins` — are **not names**. They
are the diagnostic vocabulary and a TV title, and this is the single most common failure on
clinical text: the model has never seen these words and guesses `PERSON`. That is what the
skiplist in section 5 is for. Note also that `Brandon Sanderson` and `Sanderson` are
reported as two separate entities for the same person.

Now the ALL-CAPS experiment, on the five posts that contain at least one name:

In [ ]:
with_names = [p for p, d in zip(posts, docs)
              if any(e.label_ == "PERSON" for e in d.ents)]

def person_set(texts):
    return {e.text for d in nlp.pipe(texts) for e in d.ents if e.label_ == "PERSON"}

original = person_set(with_names)
uppercased = person_set([p.upper() for p in with_names])

print(f"{len(with_names)} posts contain a name")
print(f"original  : {len(original):2d} -> {sorted(original)}")
print(f"UPPERCASED: {len(uppercased):2d} -> {sorted(uppercased)}")

Nine distinct spans become seven, and the change is not a clean subset:

| | found |
| --- | --- |
| original | `Asperger`, `Brandon Sanderson`, `Nicholas Hoult`, `Sanderson`, `Sid`, `Skins`, `Yada Yada`, `aspergers`, `dyspraxia` |
| uppercased | `ASPERGER`, `BRANDON SANDERSON`, `DYSPRAXIA`, `I'VE`, `LIP`, `NICHOLAS HOULT`, `SANDERSON` |

Two real names (`Sid`, `Yada Yada`) are **lost**, and two new false positives appear
(`I'VE`, `LIP`). Uppercasing destroys the capitalization signal the models lean on, so both
precision and recall move — which is also why **title-casing your text to boost recall is
a bad trade**: you gain names and gain even more noise.

### 8.2 The rule-based pass

The rules exploit something the models ignore: in record-style text, names sit in
predictable places. Three ingredients:

1. **Anchors** — titles (`Dr.`, `Lic.`, `Enf.`) and field labels (`Paciente:`,
   `Nombre y Apellido:`) are followed by a name. This works in ALL-CAPS text, where the
   models do not.
2. **A guard list** — the tokens a name may never absorb: articles, prepositions, clinical
   vocabulary, institution words, month and weekday names, and the title/label words
   themselves. Without the last group the pass eats the next field's label.
3. **Extension to a fixed point** — grow each known name over adjacent capitalized tokens,
   repeatedly, until nothing changes. Iteration is what handles two given names *and* two
   surnames: `María` → `María José` → `María José García` → `María José García Pérez`
   needs several passes. Cap the length (five tokens is a realistic ceiling) so a runaway
   match cannot swallow a sentence.

The same code also **trims** spans, which is what fixes the newline bleed and the absorbed
titles above.

In [ ]:
import re

CAP = r"[A-ZÁÉÍÓÚÑ][a-záéíóúñ]+"      # a capitalized, accent-aware word
MAX_NAME_TOKENS = 5                    # two given names + two surnames, plus slack

SPANISH = {
    "titles": ["Dr", "Dra", "Lic", "Licenciada", "Licenciado", "Enf", "Sr", "Sra",
               "Srta", "Prof", "Psic"],
    "labels": ["Nombre y Apellido", "Nombre", "Apellido", "Paciente", "Acompañante",
               "Responsable", "Madre", "Padre", "Tutor", "Derivado por"],
    "stopwords": {"El", "La", "Los", "Las", "Un", "Una", "Se", "Su", "Sus", "Del", "De",
                  "Y", "En", "Con", "Por", "Para", "No", "Si", "Hospital", "Centro",
                  "Salud", "Mental", "Guardia", "Servicio", "Femenino", "Masculino",
                  "Lunes", "Martes", "Enero", "Febrero"},
}

ENGLISH = {
    "titles": ["Dr", "Doctor", "Mr", "Mrs", "Ms", "Prof", "Nurse", "Therapist"],
    "labels": ["Name", "Patient", "Client", "Contact", "Referred by"],
    "stopwords": {"I", "The", "This", "That", "My", "But", "And", "So", "It", "He", "She",
                  "They", "We", "You", "A", "An", "If", "When", "What", "Just", "Then",
                  "Now", "There", "Here", "Also", "Autism", "Autistic", "Asperger",
                  "Aspergers", "Anxiety", "Depression", "ADHD", "OCD", "PTSD", "Monday",
                  "January", "University", "College", "School", "Please", "Help",
                  "Thanks", "Edit"},
}


def guard(cfg):
    """Tokens a name may never absorb, upper-cased for case-insensitive matching.

    Title and label words are guards too: without them the extension swallows the
    next field's label ("María José García Pérez Acompañante").
    """
    words = set(cfg["stopwords"]) | set(cfg["titles"])
    for label in cfg["labels"]:
        words |= set(label.split())
    return {w.upper() for w in words}


def _trim(tokens, blocked):
    """Keep the leading run of name-ish tokens, stopping at the first guard word."""
    kept = []
    for token in tokens[:MAX_NAME_TOKENS]:
        if token.upper().strip(".,;:") in blocked:
            break
        kept.append(token)
    return kept


def anchors_from_rules(text, cfg):
    """Names sitting after a title or a field label, in mixed case *or* ALL CAPS."""
    titles = "|".join(re.escape(t) for t in cfg["titles"])
    labels = "|".join(re.escape(la) for la in cfg["labels"])
    patterns = [
        re.compile(rf"\b(?:{titles})\.?\s+({CAP}(?:\s+{CAP})*)"),        # Dra. Ana Lucía
        re.compile(rf"\b(?:{labels})\s*:\s*([^\n,;.]+)"),                 # Paciente: ...
        re.compile(                                                        # ALL-CAPS variant
            rf"\b(?:(?:{titles.upper()})\.?|(?:{labels.upper()})\s*:)\s+"
            rf"([A-ZÁÉÍÓÚÑ]{{2,}}(?:\s+[A-ZÁÉÍÓÚÑ]{{2,}})*)"
        ),
    ]
    blocked = guard(cfg)
    found = set()
    for pattern in patterns:
        for match in pattern.finditer(text):
            tokens = _trim(match.group(1).strip().split(), blocked)
            if tokens:
                found.add(" ".join(tokens))
    return found


def extend_spans(text, known, cfg, max_passes=5):
    """Trim the seeds, then grow them over adjacent capitalized tokens to a fixed point."""
    blocked = guard(cfg)

    # Clean the seeds first: model spans run across line breaks (picking up the next
    # field's label) and, in ALL-CAPS text, keep going into ordinary words.
    spans = set()
    for seed in known:
        tokens = _trim(seed.split("\n")[0].strip().split(), blocked)
        if tokens:
            spans.add(" ".join(tokens))

    for _ in range(max_passes):
        grown = set()
        for name in spans:
            # In ALL-CAPS text the mixed-case pattern never matches: take the token
            # shape from the seed itself.
            shape = r"[A-ZÁÉÍÓÚÑ]{2,}" if name.isupper() else CAP
            pattern = re.compile(
                rf"((?:{shape}[ \t]+)*){re.escape(name)}((?:[ \t]+{shape})*)"
            )
            for match in pattern.finditer(text):
                left = _trim(match.group(1).split()[::-1], blocked)[::-1]
                right = _trim(match.group(2).split(), blocked)
                candidate = " ".join([*left, name, *right]).strip()
                if candidate != name and len(candidate.split()) <= MAX_NAME_TOKENS:
                    grown.add(candidate)
        if not grown - spans:
            break
        spans |= grown

    # Keep maximal spans only, so "Sanderson" disappears in favour of "Brandon Sanderson".
    return {s for s in spans if not any(s != other and s in other for other in spans)}


def recover_names(text, seeds, cfg):
    """Full pass: model seeds + rule anchors, trimmed and extended."""
    return extend_spans(text, set(seeds) | anchors_from_rules(text, cfg), cfg)

### 8.3 What it does and does not add

Run it over the Reddit posts, seeded with the spaCy spans:

In [ ]:
added = {}
for post, doc in zip(posts, docs):
    seeds = {e.text for e in doc.ents if e.label_ == "PERSON"}
    if not seeds:
        continue
    extra = recover_names(post, seeds, ENGLISH) - seeds
    if extra:
        added[post[:60]] = extra

print(added or "no spans added")

**Nothing is added** — and that is the correct result. These are conversational Reddit
posts: no titles, no field labels, and the names that are present are already whole. A
rule pass that stayed silent here is a rule pass that will not flood your record-style
data with false positives either.

The rules earn their keep on record-style text. Here is the same pass on a short synthetic
record (invented names, Spanish, the layout that breaks the models), in both mixed case
and ALL CAPS:

In [ ]:
DEMO = """Informe de guardia
Paciente: María José García Pérez
Acompañante: Juan Carlos Gómez Fernández
Evaluada por la Dra. Ana Lucía Ramírez en el Hospital San Roque.
La paciente refiere ansiedad. Se deriva a Salud Mental.
"""

nlp_es = spacy.load("es_core_news_lg")   # python -m spacy download es_core_news_lg

for label, text in (("mixed case", DEMO), ("ALL CAPS", DEMO.upper())):
    doc = nlp_es(text)
    seeds = {e.text for e in doc.ents if e.label_ == "PER"}
    print(f"--- {label} ---")
    print(f"  model spans   : {sorted(seeds)}")
    print(f"  rule anchors  : {sorted(anchors_from_rules(text, SPANISH))}")
    print(f"  after recovery: {sorted(recover_names(text, seeds, SPANISH))}")

Both variants converge on the three names, and the artifacts are gone:

```
--- mixed case ---
  model spans   : ['Juan Carlos Gómez Fernández\nEvaluada por la Dra. Ana Lucía Ramírez',
                   'María José García Pérez\nAcompañante']       ← bled across two lines
  after recovery: ['Ana Lucía Ramírez', 'Juan Carlos Gómez Fernández',
                   'María José García Pérez']                    ✓

--- ALL CAPS ---
  model spans   : ['ANA LUCÍA RAMÍREZ EN EL', 'JUAN CARLOS GÓMEZ FERNÁNDEZ',
                   'MARÍA JOSÉ GARCÍA PÉREZ']                    ← ran into ordinary words
  after recovery: ['ANA LUCÍA RAMÍREZ', 'JUAN CARLOS GÓMEZ FERNÁNDEZ',
                   'MARÍA JOSÉ GARCÍA PÉREZ']                    ✓
```

Note `ANA LUCÍA RAMÍREZ`: the model ran the span into `EN EL`, and the guard list stopped
it at `EN` while the `Dra.` anchor confirmed where the name began. `HOSPITAL SAN ROQUE` was
never absorbed because `Hospital` is a guard word — an institution, not a person.

### 8.4 Folding the recovered names back in

`anonymize-pii` has no hook for an extra deny list, so apply the recovered spans as a
second replacement pass over its output. Two rules matter: seed from the **original** text
(that is where the names still are), and replace **longest span first**, so
`María José García Pérez` is masked before a bare `María` can cut it in half.

In a real run take your seeds from `Iterator.json` instead of re-running spaCy — the
pipeline already did the detection:

In [ ]:
def person_seeds_from_iterator(iterator, doc_id, engines=("spacy", "stanza", "GLiNER")):
    """Pull the spans the pipeline labelled PERSON for one document."""
    return {
        text
        for engine in engines
        for text, (entity_type, _score) in iterator.get(doc_id, {}).get(engine, {}).items()
        if entity_type == "PERSON"
    }


def _fragments(name, blocked):
    """Contiguous sub-spans of a name, longest first.

    The full span is often no longer in the anonymized text: if the pipeline caught
    only the first given name, "María José García Pérez" survives as
    "<PERSON> José García Pérez", so the leftover has to be matched on its own.
    """
    tokens = name.split()
    spans = [
        tokens[start:end]
        for size in range(len(tokens), 0, -1)
        for start in range(0, len(tokens) - size + 1)
        for end in (start + size,)
    ]
    return [
        " ".join(span)
        for span in spans
        if not (len(span) == 1 and (len(span[0]) < 3 or span[0].upper() in blocked))
    ]


def apply_recovered_names(original, anonymized, seeds, cfg, replacement="<PERSON>"):
    """Mask rule-recovered names, including ones the pipeline only partly masked."""
    recovered = recover_names(original, seeds, cfg)
    blocked = guard(cfg)
    masked = set()

    for name in sorted(recovered, key=len, reverse=True):
        fragments = _fragments(name, blocked)
        for _ in range(len(fragments)):          # bounded: each pass removes text
            hit = next(
                (f for f in fragments if f in anonymized.replace(replacement, "")),
                None,
            )
            if hit is None:
                break
            anonymized = anonymized.replace(hit, replacement)
            masked.add(hit)

    return anonymized, {"recovered": recovered, "masked": masked}


# reports = the {id: text} dict you fed the pipeline; anonymized = its output
patched, report_log = {}, {}
for doc_id, anon_text in anonymized.items():
    seeds = person_seeds_from_iterator(iterator, doc_id)
    patched[doc_id], report_log[doc_id] = apply_recovered_names(
        reports[doc_id], anon_text, seeds, SPANISH
    )

still_leaking = {k: v["masked"] for k, v in report_log.items() if v["masked"]}
print(f"{len(still_leaking)} documents had names the pipeline left in the clear")
for doc_id, spans in list(still_leaking.items())[:10]:
    print(f"  {doc_id}: {sorted(spans)}")

Review `report_log` before trusting `patched`. Each entry records what the rules decided was
a name (`recovered`) and what actually still needed masking (`masked`) — a rule pass is only
as good as its guard list. Add every false positive you find to `stopwords`, then re-run:
the same review loop as section 5, and worth two or three iterations on a new corpus.

Use `patched` as the text you send onward, and keep `report_log` as evidence of what the
second pass caught.

## 9. Feed the anonymized text into `llm_tracker`

Turn `Anonymized_Reports.json` back into a CSV. `analyze_csv()` builds each document ID
by joining two columns with `_`, so keeping your anonymize-pii IDs in one of them
preserves traceability back to the source records.

In [ ]:
anon_df = pd.DataFrame(
    {
        "source": "anonymized",            # first half of the llm_tracker document ID
        "doc_id": list(anonymized),        # second half — keeps the anonymize-pii IDs
        "text": list(anonymized.values()),
    }
)

anon_csv = Path("anonymized_documents.csv")
anon_df.to_csv(anon_csv, index=False)
anon_df.head()

In [ ]:
from llm_tracker import AnalyzerConfig, LLMTrackerAnalyzer

config = AnalyzerConfig(
    api_key="",                                    # your OpenRouter key, or a path to a .env
    model_name="google/gemini-3-flash-preview",
)
analyzer = LLMTrackerAnalyzer(config=config)

results_llm, metadata_llm, errors_llm = analyzer.analyze_csv(
    csv_path=str(anon_csv),
    codebook_path="codebook.json",
    text_column="text",
    subreddit_column="source",     # document IDs come out as "anonymized_<doc_id>"
    author_column="doc_id",
    output_dir="LLM_coding",
)

From here on, continue with [`tutorial.ipynb`](../tutorial.ipynb) — comparison against human
coding, summary tables, and metrics.

---

## Before you send anything to an API

- [ ] Every document went through the anonymizer (no partially processed batches).
- [ ] You reviewed the low-confidence detections and the single-engine detections.
- [ ] If your text is record-style or upper-case, you ran a rule pass over the
      names (section 8) and reviewed what it recovered.
- [ ] Known identifiers from your source no longer appear in the output.
- [ ] Structured / controlled-vocabulary fields were de-identified with your own rules,
      not with NER.
- [ ] Free-text fields you did not intend to send are excluded from the CSV.
- [ ] A human read at least a sample of the anonymized documents end to end.
- [ ] The un-anonymized source stays out of synced, shared, and version-controlled
      directories.